# 2. Extracting Pain Points & Expectations
### Input: output/docx_sections/{FocusGroup}.docx
### Output: output/challenges.docx & output/expectations.docx

In [1]:
pip install csv

ERROR: Could not find a version that satisfies the requirement csv (from versions: none)
ERROR: No matching distribution found for csv
Note: you may need to restart the kernel to use updated packages.


In [2]:
from docx import Document
import pandas as pd
import csv
import os

In [3]:
input_path = '../output/docx_sections'
# create the output CSV files if they do not exist
if not os.path.exists('../output/challenges.csv') or not os.path.exists('../output/expectations.csv'):
    os.makedirs('../output', exist_ok=True)
    if not os.path.exists('../output/challenges.csv'):
        with open('../output/challenges.csv', 'w', newline='', encoding='utf-8') as csvfile:
            writer = csv.writer(csvfile)
            writer.writerow(['focus_group', 'pain_points'])
    if not os.path.exists('../output/expectations.csv'):
        with open('../output/expectations.csv', 'w', newline='', encoding='utf-8') as csvfile:
            writer = csv.writer(csvfile)
            writer.writerow(['focus_group', 'expectations'])

In [4]:
#Extract pain-points and expectations from the docx file and write to CSV files
def extract_challenges_and_expectations(input_path, output_dir):
    """Scan .docx files in `input_path`, extract content under headings 'Pain Points'
    and 'Expectations', and append rows to CSV files in `output_dir`. Returns a dict
    with lists of extracted tuples (focus_group, text)."""
    os.makedirs(output_dir, exist_ok=True)
    challenges_csv = os.path.join(output_dir, 'challenges.csv')
    expectations_csv = os.path.join(output_dir, 'expectations.csv')

    # Ensure CSV headers exist
    if not os.path.exists(challenges_csv):
        with open(challenges_csv, 'w', newline='', encoding='utf-8') as csvfile:
            writer = csv.writer(csvfile)
            writer.writerow(['focus_group', 'pain_points'])
    if not os.path.exists(expectations_csv):
        with open(expectations_csv, 'w', newline='', encoding='utf-8') as csvfile:
            writer = csv.writer(csvfile)
            writer.writerow(['focus_group', 'expectations'])

    results = {'pain_points': [], 'expectations': []}

    # Go through each file in the input dir and extract the sections
    for filename in os.listdir(input_path):
        if not filename.lower().endswith('.docx'):
            continue
        doc = Document(os.path.join(input_path, filename))
        focus_group_name = os.path.splitext(filename)[0]
        paras = doc.paragraphs
        i = 0
        while i < len(paras):
            para = paras[i]
            style_name = ''
            try:
                if para.style is not None:
                    style_name = para.style.name
            except Exception:
                style_name = ''

            if style_name.startswith('Heading 3') and 'Pain-Point:' in para.text:
                # collect following non-heading paragraphs as the pain point description
                j = i + 1
                collected = []
                while j < len(paras):
                    next_para = paras[j]
                    next_style = ''
                    try:
                        if next_para.style is not None:
                            next_style = next_para.style.name
                    except Exception:
                        next_style = ''
                    if next_style.startswith('Heading'):
                        break
                    if next_para.text.strip():
                        collected.append(next_para.text.strip())
                    j += 1
                desc = '\n'.join(collected).strip() if collected else para.text.strip()
                # Split description on newlines and write each non-empty line as a separate CSV row
                lines = [line.strip() for line in desc.splitlines() if line.strip()]
                for line in lines:
                    results['pain_points'].append((focus_group_name, line))
                    with open(challenges_csv, 'a', newline='', encoding='utf-8') as csvfile:
                        writer = csv.writer(csvfile)
                        writer.writerow([focus_group_name, line])
                i = j
                continue

            if style_name.startswith('Heading 3') and 'Expectation:' in para.text:
                j = i + 1
                collected = []
                while j < len(paras):
                    next_para = paras[j]
                    next_style = ''
                    try:
                        if next_para.style is not None:
                            next_style = next_para.style.name
                    except Exception:
                        next_style = ''
                    if next_style.startswith('Heading'):
                        break
                    if next_para.text.strip():
                        collected.append(next_para.text.strip())
                    j += 1
                desc = '\n'.join(collected).strip() if collected else para.text.strip()
                # Split expectation description on newlines and write each non-empty line as a separate CSV row
                lines = [line.strip() for line in desc.splitlines() if line.strip()]
                for line in lines:
                    results['expectations'].append((focus_group_name, line))
                    with open(expectations_csv, 'a', newline='', encoding='utf-8') as csvfile:
                        writer = csv.writer(csvfile)
                        writer.writerow([focus_group_name, line])
                i = j
                continue

            i += 1

    print(f"Extracted {len(results['pain_points'])} pain points and {len(results['expectations'])} expectations")
    return results

# Run the extractor on the docx sections folder
results = extract_challenges_and_expectations(input_path, '../output')

pd.DataFrame(results['pain_points'], columns=['focus_group', 'pain_points']).to_csv('../output/challenges.csv', index=False, encoding='utf-8-sig')
pd.DataFrame(results['expectations'], columns=['focus_group', 'expectations']).to_csv('../output/expectations.csv', index=False, encoding='utf-8-sig')

Extracted 424 pain points and 433 expectations
